# 01 - Exploratory Data Analysis (EDA)
## Phân tích lỗi sản xuất & Dự đoán lỗi – UCI AI4I 2020 Predictive Maintenance

### Mục tiêu
1. Mô tả bối cảnh bài toán và dữ liệu
2. Tạo Data Dictionary
3. EDA: phân bố, tương quan, phát hiện bất thường
4. Nhận diện rủi ro: mất cân bằng lớp, data leakage, thiếu dữ liệu

## 1. Đặt vấn đề

**Bối cảnh:** Trong sản xuất công nghiệp, việc dự đoán lỗi máy móc trước khi xảy ra (Predictive Maintenance) giúp giảm thiểu thời gian ngừng máy, tiết kiệm chi phí bảo trì và nâng cao hiệu suất sản xuất.

**Dataset:** UCI AI4I 2020 Predictive Maintenance Dataset – 10,000 quan sát từ cảm biến máy công nghiệp, bao gồm 5 loại lỗi: Tool Wear (TWF), Heat Dissipation (HDF), Power (PWF), Overstrain (OSF), Random (RNF).

**Mục tiêu:**
- **Khai phá tri thức:** Tìm pattern/luật kết hợp liên quan lỗi; Phân cụm máy theo hành vi
- **Phân lớp:** Dự đoán Machine failure (binary) – xử lý imbalance
- **Hồi quy:** Dự đoán Tool wear [min]
- **Time series:** Coi UDI ≈ time index, dùng ARIMA/lag features
- **Bán giám sát:** Thử nghiệm kịch bản thiếu nhãn (5/10/20%)

**Tiêu chí thành công:**
- Classification: F1 ≥ 0.6, PR-AUC cao (do imbalance)
- Regression: MAE, RMSE thấp
- Mining: Tìm được insight hành động
- Semi-supervised: Learning curve cho thấy cải thiện so với supervised-only

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data.loader import load_params, load_raw_data, validate_schema, create_data_dictionary, get_data_summary
from src.visualization.plots import (
    plot_target_distribution, plot_failure_types, plot_numeric_distributions,
    plot_correlation_matrix, plot_feature_vs_target, save_fig
)

# Load config
params = load_params('../configs/params.yaml')
print('Config loaded ✓')

## 2. Load & Validate Data

In [ ]:
# Load raw data
df = load_raw_data(path='../data/raw/ai4i2020.csv')

# Validate schema
validate_schema(df, params)

# Basic info
print(f'\nShape: {df.shape}')
print(f'\nDtypes:\n{df.dtypes}')
df.head()

## 3. Data Dictionary

In [ ]:
# Tạo Data Dictionary
data_dict = create_data_dictionary(df)
data_dict.style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('text-align', 'left')]}]
)

## 4. Thống kê mô tả

In [ ]:
# Thống kê mô tả
print('=== Thống kê cơ bản ===')
display(df.describe().round(2))

print('\n=== Missing values ===')
print(df.isnull().sum())

print(f'\n=== Duplicates (excluding ID): {df.drop(columns=["UDI", "Product ID"]).duplicated().sum()} ===')

# Summary
summary = get_data_summary(df)
print(f"\nTotal missing: {sum(summary['missing'].values())}")
print(f"Duplicates: {summary['duplicates']}")

## 5. EDA – Phân bố Target (Mất cân bằng lớp)

In [ ]:
# 5.1 Phân bố Machine failure (target)
print('Machine failure distribution:')
print(df['Machine failure'].value_counts())
print(f'\nFailure rate: {df["Machine failure"].mean()*100:.2f}%')
print(f'Imbalance ratio: 1:{int((1-df["Machine failure"].mean())/df["Machine failure"].mean())}')

fig = plot_target_distribution(df['Machine failure'])
save_fig(fig, '01_target_distribution', '../outputs/figures')
plt.show()

In [ ]:
# 5.2 Phân bố từng loại lỗi
failure_cols = params['data']['failure_types']
print('Failure type counts:')
for col in failure_cols:
    print(f'  {col}: {df[col].sum()} ({df[col].mean()*100:.2f}%)')

fig = plot_failure_types(df, failure_cols)
save_fig(fig, '01_failure_types', '../outputs/figures')
plt.show()

In [ ]:
# 5.3 Multi-failure analysis
df['n_failures'] = df[failure_cols].sum(axis=1)
print('Number of simultaneous failures per observation:')
print(df['n_failures'].value_counts().sort_index())

# Failure co-occurrence matrix
cooccur = df[failure_cols].T.dot(df[failure_cols])
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cooccur, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_title('Failure Type Co-occurrence')
save_fig(fig, '01_failure_cooccurrence', '../outputs/figures')
plt.show()

## 6. EDA – Phân bố biến số

In [ ]:
# 6.1 Histogram + Boxplot cho từng biến số
numeric_cols = params['data']['numeric_features']
fig = plot_numeric_distributions(df, numeric_cols)
save_fig(fig, '01_numeric_distributions', '../outputs/figures')
plt.show()

In [ ]:
# 6.2 Phân bố biến phân loại Type
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
type_counts = df['Type'].value_counts()
axes[0].bar(type_counts.index, type_counts.values, color=['#3498db', '#e67e22', '#2ecc71'])
axes[0].set_title('Product Type Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(type_counts.values):
    axes[0].text(i, v+50, str(v), ha='center', fontweight='bold')

# Failure rate by Type
fr_by_type = df.groupby('Type')['Machine failure'].mean() * 100
axes[1].bar(fr_by_type.index, fr_by_type.values, color=['#3498db', '#e67e22', '#2ecc71'])
axes[1].set_title('Failure Rate by Product Type (%)')
axes[1].set_ylabel('Failure Rate (%)')
for i, v in enumerate(fr_by_type.values):
    axes[1].text(i, v+0.1, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
save_fig(fig, '01_type_distribution', '../outputs/figures')
plt.show()

## 7. EDA – Tương quan & So sánh theo Target

In [ ]:
# 7.1 Correlation matrix
fig = plot_correlation_matrix(df, numeric_cols + ['Machine failure'] + failure_cols)
save_fig(fig, '01_correlation_matrix', '../outputs/figures')
plt.show()

In [ ]:
# 7.2 Feature vs Target
for col in numeric_cols:
    fig = plot_feature_vs_target(df, col)
    save_fig(fig, f'01_feature_{col.replace(" ", "_").replace("[", "").replace("]", "")}', '../outputs/figures')
    plt.show()

In [ ]:
# 7.3 Pairplot (sample to avoid slowness)
sample = df.sample(n=2000, random_state=42)
g = sns.pairplot(sample, vars=numeric_cols, hue='Machine failure',
                 palette={0: '#2ecc71', 1: '#e74c3c'}, diag_kind='kde',
                 plot_kws={'alpha': 0.4, 's': 15})
g.figure.suptitle('Pairplot – Features by Machine Failure', y=1.02)
save_fig(g.figure, '01_pairplot', '../outputs/figures')
plt.show()

## 8. Phân tích rủi ro

In [ ]:
print('=== PHÂN TÍCH RỦI RO ===')
print()

# 1. Mất cân bằng lớp
failure_rate = df['Machine failure'].mean()
print(f'1. MẤT CÂN BẰNG LỚP:')
print(f'   - Failure rate: {failure_rate*100:.2f}%')
print(f'   - Ratio Normal:Failure = {int((1-failure_rate)/failure_rate)}:1')
print(f'   → Cần dùng: class_weight="balanced", SMOTE, PR-AUC thay vì accuracy')
print()

# 2. Data leakage
print('2. DATA LEAKAGE:')
print('   - Cột TWF/HDF/PWF/OSF/RNF là THÀNH PHẦN của Machine failure')
print('   → KHÔNG được dùng làm feature cho classification Machine failure!')
print('   → Có thể dùng cho multi-label hoặc phân tích loại lỗi')
print()

# 3. Missing data
print('3. MISSING DATA:')
print(f'   - Total missing values: {df.isnull().sum().sum()}')
print('   → Dataset sạch, không có missing')
print()

# 4. Temporal assumption
print('4. GIẢ ĐỊNH TEMPORAL:')
print('   - UDI là ID tăng dần, KHÔNG phải timestamp thực')
print('   - Khi dùng ARIMA/lag features: cần nêu rõ giả định UDI ≈ thời gian')
print('   - Train/test split theo thứ tự UDI (không shuffle) cho time series')
print()

# 5. Kiểm tra leakage: correlation failure types vs target
print('5. CORRELATION FAILURE TYPES vs TARGET:')
for col in failure_cols:
    corr = df[col].corr(df['Machine failure'])
    print(f'   {col} ↔ Machine failure: {corr:.4f}')

# Clean up temp column
df = df.drop(columns=['n_failures'], errors='ignore')

## 9. Kết luận EDA

### Findings chính:
1. **Mất cân bằng lớp nghiêm trọng:** Chỉ 3.39% failure → cần class_weight, SMOTE, đánh giá bằng F1/PR-AUC
2. **5 loại lỗi với cơ chế khác nhau:** HDF (115) phổ biến nhất, RNF (19) hiếm nhất
3. **Data leakage tiềm ẩn:** TWF/HDF/PWF/OSF/RNF là thành phần của target → loại bỏ khi train classifier
4. **Tương quan rõ:** Torque & Rotational speed tương quan nghịch mạnh; temp_diff liên quan HDF
5. **Không có missing/duplicate:** Dataset sạch
6. **Outliers nhẹ:** Rotational speed có vài giá trị cao bất thường